# Iteración, recursos y concurrencia

**Unidad 1 · Sesión 3**

Iteradores y generadores para procesar secuencias; context managers para delimitar
recursos; `asyncio` para coordinar operaciones que pasan parte de su tiempo esperando.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Iterables e iteradores |
| 2 | Funciones generadoras y `yield` |
| 3 | Expresiones generadoras y consumo parcial |
| 4 | Composición y errores diferidos |
| 5 | Lectura de archivos con `with` |
| 6 | Context managers con clases |
| 7 | Context managers con `@contextmanager` |
| 8 | Generadores y duración de los recursos |
| 9 | Procesamiento de un archivo y medición de memoria |
| 10 | Práctica independiente |
| 11 | Corrutinas, `async` y `await` |
| 12 | Ejecución secuencial y tareas concurrentes |
| 13 | Cancelación y tiempos límite |
| 14 | Context managers asíncronos |
| 15 | Generadores asíncronos y `async for` |
| 16 | Integración y práctica de concurrencia |

## Objetivos

- Diferenciar un iterable de un iterador y reconocer cuándo se agota.
- Construir transformaciones que produzcan valores bajo demanda.
- Garantizar el cierre de archivos al terminar, interrumpir o fallar un recorrido.
- Comparar la memoria de una lista completa con la de un procesamiento incremental.
- Coordinar tareas de E/S con `asyncio` y conservar el orden de los resultados.
- Comprobar cancelación, tiempos límite y cierre de recursos asíncronos.

## Preparación

Python 3.12 o posterior. Ejecuta las celdas en orden. Los ejemplos utilizan la
biblioteca estándar y crean sus archivos de práctica en directorios temporales.
Cada ejercicio se resuelve en su celda y conserva sus comprobaciones.

In [ ]:
import sys
from collections.abc import Iterable, Iterator
from contextlib import contextmanager, closing
from itertools import islice
from pathlib import Path
from tempfile import TemporaryDirectory

assert sys.version_info >= (3, 12), "Use Python 3.12 or newer"

## 1. Iterables e iteradores

Un **iterable** puede proporcionar un iterador mediante `iter(objeto)`. Listas,
cadenas y rangos son ejemplos. Un **iterador** conserva el estado de un recorrido:
`next(iterador)` obtiene el siguiente elemento y avanza su posición.

Una lista puede producir varios iteradores independientes. El iterador no vuelve
al inicio por entrar en otro `for`. Cuando termina, `next` lanza `StopIteration`.
Un `for` atiende esa excepción y finaliza el recorrido.

In [ ]:
names = ["north", "south", "west"]
first_iterator = iter(names)
second_iterator = iter(names)

print(next(first_iterator))
print(next(first_iterator))
print(next(second_iterator))
assert list(first_iterator) == ["west"]
assert list(first_iterator) == []
assert names == ["north", "south", "west"]

In [ ]:
iterator = iter([10])
print(next(iterator))
try:
    next(iterator)
except StopIteration:
    print("Iterator exhausted")

print(next(iterator, "No more values"))
assert iter(iterator) is iterator

### Ejercicio 1 · Estado de un recorrido

Antes de ejecutar, escribe qué valores producirán las cuatro llamadas. Después
crea otro iterador sobre la misma lista y comprueba que comienza desde el principio.

In [ ]:
values = [4, 8, 12]
value_iterator = iter(values)
# Predict: next(value_iterator), list(value_iterator), list(value_iterator), values.

### El protocolo de iteración

`__iter__` proporciona el iterador y `__next__` produce el siguiente valor. Esta
clase implementa ambos métodos para mostrar el estado que un `for` consume.
Un mismo objeto puede ser iterable e iterador, como ocurre aquí.

In [ ]:
class Countdown:
    def __init__(self, start: int):
        self.current = start

    def __iter__(self) -> "Countdown":
        return self

    def __next__(self) -> int:
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value


countdown = Countdown(3)
assert list(countdown) == [3, 2, 1]
assert list(countdown) == []

## 2. Funciones generadoras y `yield`

Una función que contiene `yield` produce un generador al llamarla. Su cuerpo
comienza a ejecutarse cuando se solicita el primer valor. `yield` entrega un valor
y suspende la función, conservando sus variables y su posición.

La siguiente petición continúa después del `yield`. Un `return` sin valor, o llegar
al final de la función, termina el recorrido. En una función generadora no debemos
lanzar `StopIteration` directamente para terminar.

In [ ]:
def countdown_values(start: int) -> Iterator[int]:
    current = start
    while current > 0:
        print(f"Producing {current}")
        yield current
        current -= 1
    print("Finished")


countdown_generator = countdown_values(3)
print("Generator created")
print("First value:", next(countdown_generator))
print("Remaining values:", list(countdown_generator))

### `return` y `yield`

Una función normal con `return` entrega su resultado al terminar. Una función
generadora puede entregar varios valores en distintas solicitudes. La anotación
`Iterator[int]` describe los valores que el consumidor obtiene; no significa que
cada llamada a la función devuelva un entero.

In [ ]:
def build_squares(limit: int) -> list[int]:
    return [number * number for number in range(limit)]


def generate_squares(limit: int) -> Iterator[int]:
    for number in range(limit):
        yield number * number


assert build_squares(4) == [0, 1, 4, 9]
assert list(generate_squares(4)) == build_squares(4)

### Ejercicio 2 · Generar sin construir una lista

Escribe `even_numbers(stop)` para producir pares desde cero hasta `stop`, sin
incluirlo. Usa `yield`. Comprueba límites 0, 1 y 7. No uses una lista como acumulador.

In [ ]:
# Define even_numbers(stop: int) -> Iterator[int].

## 3. Expresiones generadoras y consumo parcial

Una expresión entre paréntesis puede producir valores bajo demanda. La expresión
con corchetes construye una lista. En ambos casos el cálculo es el mismo, pero
cambian el momento en que se realiza y la conservación de los resultados.

`sum`, `list` y los ciclos consumen el iterador. Un generador agotado no conserva
los valores para otro recorrido. Si necesitas volver a procesarlos, crea un nuevo
generador o guarda una colección, según el tamaño y el uso.

In [ ]:
squares_list = [number * number for number in range(5)]
squares_generator = (number * number for number in range(5))

assert sum(squares_generator) == 30
assert sum(squares_generator) == 0
assert sum(squares_list) == 30
assert sum(squares_list) == 30

### `islice`: limitar el consumo

`islice(iterador, cantidad)` entrega como máximo esa cantidad de elementos. Permite
consumir una parte de una secuencia, incluso si la fuente no tiene fin. El iterador
original queda situado después del último elemento consumido.

In [ ]:
def increasing_numbers(start: int = 0) -> Iterator[int]:
    current = start
    while True:
        yield current
        current += 1


numbers = increasing_numbers(10)
assert list(islice(numbers, 4)) == [10, 11, 12, 13]
assert next(numbers) == 14
numbers.close()

### Ejercicio 3 · Dos recorridos sobre un generador

Este código calcula dos resúmenes sobre el mismo generador. Predice el resultado
y corrígelo sin materializar todos los valores: calcula suma y cantidad en un solo
recorrido. Para cinco valores, deben resultar 10 y 5.

In [ ]:
sample_values = (number for number in range(5))
total = sum(sample_values)
count = sum(1 for _ in sample_values)
print(total, count)
# Recreate the generator and calculate both results in one pass.

## 4. Composición y errores diferidos

Podemos conectar una fuente con funciones que transforman y filtran registros.
Cada función recibe un iterable y devuelve un iterador. El consumidor final
solicita los resultados y hace avanzar las funciones anteriores.

Usaremos líneas con el formato `station;value`. Un registro válido tiene un nombre
de estación no vacío y un número finito. Las líneas vacías se ignoran. El formato
no admite campos entrecomillados ni separadores dentro del nombre.

In [ ]:
from math import isfinite

Measurement = tuple[str, float]

def parse_measurements(lines: Iterable[str]) -> Iterator[Measurement]:
    """Read station;value lines; ignore blank lines and report physical line numbers."""
    for line_number, line in enumerate(lines, start=1):
        if not line.strip():
            continue
        parts = line.strip().split(";")
        if len(parts) != 2 or not parts[0].strip():
            raise ValueError(f"Line {line_number}: expected station;value")
        station, raw_value = parts
        try:
            value = float(raw_value)
        except ValueError as error:
            raise ValueError(f"Line {line_number}: invalid number") from error
        if not isfinite(value):
            raise ValueError(f"Line {line_number}: value must be finite")
        yield station.strip(), value


def select_measurements(
    measurements: Iterable[Measurement], minimum: float
) -> Iterator[Measurement]:
    for station, value in measurements:
        if value >= minimum:
            yield station, value


def summarize_measurements(
    measurements: Iterable[Measurement],
) -> tuple[int, float | None]:
    count = 0
    total = 0.0
    for _, value in measurements:
        count += 1
        total += value
    return count, total / count if count else None

In [ ]:
raw_lines = ["north;18.0", "south;27.0", "west;32.0", "north;23.0"]
measurements = parse_measurements(raw_lines)
selected = select_measurements(measurements, minimum=25.0)
count, average = summarize_measurements(selected)
assert count == 2
assert average == 29.5
print(count, average)

### El consumidor determina cuánto avanza la fuente

En esta composición síncrona, crear el generador no inicia un recorrido completo.
Al pedir un resultado al filtro, este puede necesitar leer varias entradas hasta
encontrar una que cumpla la condición. No hay tareas en segundo plano.

In [ ]:
def traced_lines() -> Iterator[str]:
    for line in ["north;18.0", "south;27.0", "west;32.0"]:
        print(f"Reading: {line}")
        yield line


pipeline = select_measurements(parse_measurements(traced_lines()), 25.0)
print("Pipeline created")
print("First match:", next(pipeline))
# The last source line has not been read yet.
pipeline.close()

### Ejercicio 4 · Consumo bajo demanda

Construye una fuente con un contador o una lista de valores visitados. Usa las
lecturas 18, 27 y 32, filtra desde 25 y solicita un solo resultado. Comprueba que
la fuente visitó exactamente dos lecturas, no una ni tres.

In [ ]:
# Track visited values and request one matching measurement.

### Errores durante el consumo

Un error dentro de una función generadora puede ocurrir al solicitar el siguiente
valor, mucho después de crear el generador. La captura debe rodear el consumo que
puede fallar. Tras una excepción no manejada dentro del generador, este termina.

In [ ]:
records = parse_measurements(["north;18.0", "south;unknown", "west;32.0"])
print("Parser created")
assert next(records) == ("north", 18.0)
try:
    next(records)
except ValueError as error:
    print(error)
else:
    raise AssertionError("Invalid number was accepted")
assert list(records) == []

### Ejercicio 5 · Localizar la entrada inválida

Comprueba que `parse_measurements(["", "north;nan"])` falla en la línea 2.
Después prueba un nombre vacío y una línea con tres campos. Escribe el `try` de
forma que capture el error cuando se consumen los resultados.

In [ ]:
# Inspect the message for each invalid input.

## 5. Lectura de archivos con `with`

Un archivo abierto mantiene recursos del sistema. `with` delimita su uso y llama
al mecanismo de salida del context manager al abandonar el bloque, incluso por
una excepción. El archivo se cierra aunque el recorrido no llegue hasta el final.

`TemporaryDirectory` también es un context manager: elimina su directorio al salir.
Lo usaremos para crear archivos de práctica sin depender de rutas previas.

In [ ]:
with TemporaryDirectory() as directory:
    sample_path = Path(directory) / "measurements.txt"
    sample_path.write_text("north;18.0\nsouth;27.0\n", encoding="utf-8")
    with sample_path.open(encoding="utf-8") as file:
        print(next(file).strip())
        assert not file.closed
    assert file.closed

### Salida por excepción

La excepción sigue propagándose después del cierre. Un context manager no tiene
que ocultar el problema para liberar el recurso.

In [ ]:
with TemporaryDirectory() as directory:
    sample_path = Path(directory) / "measurements.txt"
    sample_path.write_text("north;18.0\n", encoding="utf-8")
    try:
        with sample_path.open(encoding="utf-8") as file:
            raise RuntimeError("Processing failed")
    except RuntimeError as error:
        print(error)
    assert file.closed

## 6. Context managers con clases

Un objeto compatible con `with` define `__enter__` y `__exit__`.

- `__enter__` prepara el recurso. Su retorno es el valor recibido por `as`.
- `__exit__` recibe el tipo, valor y traceback de la excepción, o tres `None`
  si la salida fue normal. Allí liberamos el recurso.
- Si `__exit__` devuelve un valor verdadero, suprime la excepción. Devolver
  `None` o `False` permite que se propague.

Esta clase abre el archivo al entrar al bloque, no al construir el objeto.
Es un ejemplo del protocolo; para abrir archivos normalmente basta `Path.open`.
Si `__enter__` falla, Python no llama a `__exit__`: cualquier recurso adquirido
parcialmente debe liberarse dentro de la propia entrada.

In [ ]:
from types import TracebackType
from typing import TextIO


class TextFile:
    def __init__(self, path: Path):
        self.path = path
        self.file: TextIO | None = None

    def __enter__(self) -> TextIO:
        self.file = self.path.open(encoding="utf-8")
        print("File opened")
        return self.file

    def __exit__(
        self,
        exception_type: type[BaseException] | None,
        exception: BaseException | None,
        traceback: TracebackType | None,
    ) -> None:
        if self.file is not None:
            self.file.close()
        print("File closed")


with TemporaryDirectory() as directory:
    sample_path = Path(directory) / "measurements.txt"
    sample_path.write_text("north;18.0\n", encoding="utf-8")
    with TextFile(sample_path) as file:
        assert file.readline().strip() == "north;18.0"
    assert file.closed

### Ejercicio 6 · Cierre y propagación

Usa `TextFile` y lanza `ValueError("Invalid measurement")` dentro del `with`.
Comprueba fuera del bloque que el archivo está cerrado y que se recibió esa misma
excepción. Explica qué cambiaría si `__exit__` devolviera `True`.

In [ ]:
# Create a temporary file and catch the exception outside with.

## 7. Context managers con `@contextmanager`

`contextlib.contextmanager` permite expresar el mismo ciclo de vida con una función
generadora. Debe ejecutar **un solo `yield`** por uso del context manager.

El código anterior al `yield` prepara el recurso. Su valor llega a `as`. Cuando
termina el bloque, la función continúa después del `yield`. Si el bloque lanza una
excepción, esta se introduce en el punto de suspensión. `finally` ejecuta el cierre
al salir normalmente o por error.

In [ ]:
@contextmanager
def open_text(path: Path) -> Iterator[TextIO]:
    file = path.open(encoding="utf-8")
    try:
        print("Entering context")
        yield file
    finally:
        file.close()
        print("Leaving context")


with TemporaryDirectory() as directory:
    sample_path = Path(directory) / "sample.txt"
    sample_path.write_text("north;18.0\n", encoding="utf-8")
    with open_text(sample_path) as file:
        print(file.readline().strip())
    assert file.closed

### Ejercicio 7 · Interrumpir la lectura

Lee solo la primera línea de un archivo con `open_text` y termina el ciclo con
`break`. Verifica que dentro del bloque `with` el archivo sigue abierto y que,
al salir de ese bloque, queda cerrado. `break` termina el ciclo, no el `with`.

In [ ]:
# Use a two-line file, break after one line, and inspect file.closed.

## 8. Generadores y duración de los recursos

Devolver un generador no prolonga automáticamente la vida del archivo del que
leerá. Esta función termina su `with` antes de que alguien consuma los datos.
El archivo ya estará cerrado al solicitar el primer resultado.

In [ ]:
def closed_file_lines(path: Path) -> Iterator[str]:
    with path.open(encoding="utf-8") as file:
        return (line.strip() for line in file)


with TemporaryDirectory() as directory:
    sample_path = Path(directory) / "sample.txt"
    sample_path.write_text("north;18.0\n", encoding="utf-8")
    lines = closed_file_lines(sample_path)
    try:
        next(lines)
    except ValueError as error:
        print(f"Expected error: {error}")
    else:
        raise AssertionError("Reading a closed file should fail")

### Un `with` dentro de una función generadora

Si el `yield` ocurre dentro del `with`, el archivo permanece abierto mientras la
función está suspendida. Esto permite leerlo, pero exige atender el consumo parcial.
Salir de un `for` con `break` no cierra por sí solo un generador conservado en una
variable. `closing` llama a su método `close()` al abandonar el bloque.

In [ ]:
def generate_file_lines(path: Path) -> Iterator[str]:
    with path.open(encoding="utf-8") as file:
        try:
            for line in file:
                yield line.strip()
        finally:
            print("Finishing file generator")


with TemporaryDirectory() as directory:
    sample_path = Path(directory) / "sample.txt"
    sample_path.write_text("north;18.0\nsouth;27.0\n", encoding="utf-8")
    with closing(generate_file_lines(sample_path)) as lines:
        print(next(lines))
    assert list(lines) == []

### El consumidor delimita el acceso

Otra opción es exponer los registros mediante un context manager. La lectura y
su consumo quedan dentro del mismo bloque. Al salir, el archivo se cierra incluso
si el iterador conserva registros pendientes.

La anotación `Iterator[Iterator[Measurement]]` tiene dos niveles: la función del
context manager entrega una vez un iterador, y ese iterador produce mediciones.

In [ ]:
@contextmanager
def open_measurements(path: Path) -> Iterator[Iterator[Measurement]]:
    """Keep the file open only inside the caller's with block."""
    with path.open(encoding="utf-8") as file:
        yield parse_measurements(file)

In [ ]:
with TemporaryDirectory() as directory:
    sample_path = Path(directory) / "sample.txt"
    sample_path.write_text("north;18.0\nsouth;27.0\n", encoding="utf-8")
    with open_measurements(sample_path) as measurements:
        first = next(measurements)
        assert first == ("north", 18.0)
    # Do not continue consuming measurements after leaving the context.

## 9. Procesamiento de un archivo y medición de memoria

Compararemos dos implementaciones del mismo cálculo. Ambas leen el mismo archivo,
filtran valores desde 25 y devuelven cantidad y promedio. La primera guarda todos
los registros en una lista. La segunda mantiene solo el registro actual y los
acumuladores del resumen.

`tracemalloc` registra asignaciones de memoria de Python. Mediremos el pico de cada
operación, no el tamaño completo del proceso. La creación del archivo queda fuera
de la medición. Los números exactos dependen del entorno.

El tamaño acotado presupone líneas de longitud acotada: una línea enorme también
requiere memoria. Guardar todos los resultados en una lista al final eliminaría
la ventaja de este recorrido incremental.

### Módulo de procesamiento

La siguiente celda escribe `streaming.py` en el directorio actual. Contiene las
funciones anteriores para utilizarlas fuera de la notebook. Si modificas ese
archivo, guarda tu versión con otro nombre antes de volver a ejecutar la celda.

In [ ]:
%%writefile streaming.py
"""Read and summarize station measurements with bounded additional memory."""

from collections.abc import Iterable, Iterator
from contextlib import contextmanager
from math import isfinite
from pathlib import Path
from typing import TextIO

Measurement = tuple[str, float]


def parse_measurements(lines: Iterable[str]) -> Iterator[Measurement]:
    """Read station;value lines; ignore blank lines and report physical line numbers."""
    for line_number, line in enumerate(lines, start=1):
        if not line.strip():
            continue
        parts = line.strip().split(";")
        if len(parts) != 2 or not parts[0].strip():
            raise ValueError(f"Line {line_number}: expected station;value")
        station, raw_value = parts
        try:
            value = float(raw_value)
        except ValueError as error:
            raise ValueError(f"Line {line_number}: invalid number") from error
        if not isfinite(value):
            raise ValueError(f"Line {line_number}: value must be finite")
        yield station.strip(), value


def select_measurements(
    measurements: Iterable[Measurement], minimum: float
) -> Iterator[Measurement]:
    for station, value in measurements:
        if value >= minimum:
            yield station, value


def summarize_measurements(
    measurements: Iterable[Measurement],
) -> tuple[int, float | None]:
    count = 0
    total = 0.0
    for _, value in measurements:
        count += 1
        total += value
    return count, total / count if count else None


@contextmanager
def open_measurements(path: Path) -> Iterator[Iterator[Measurement]]:
    """Keep the file open only inside the caller's with block."""
    with path.open(encoding="utf-8") as file:
        yield parse_measurements(file)


def write_sample(file: TextIO, repetitions: int) -> None:
    """Write fixed records incrementally; do not allocate the complete dataset."""
    if repetitions < 0:
        raise ValueError("repetitions must be nonnegative")
    for _ in range(repetitions):
        file.write("north;18.0\nsouth;27.0\nwest;32.0\nnorth;23.0\n")

In [ ]:
import importlib
import streaming

streaming = importlib.reload(streaming)

In [ ]:
def summarize_eager(path: Path) -> tuple[int, float | None]:
    with streaming.open_measurements(path) as measurements:
        all_measurements = list(measurements)
    return streaming.summarize_measurements(
        streaming.select_measurements(all_measurements, 25.0)
    )


def summarize_streaming(path: Path) -> tuple[int, float | None]:
    with streaming.open_measurements(path) as measurements:
        return streaming.summarize_measurements(
            streaming.select_measurements(measurements, 25.0)
        )

In [ ]:
import gc
import tracemalloc
from collections.abc import Callable


def measure_peak(
    operation: Callable[[], tuple[int, float | None]],
) -> tuple[tuple[int, float | None], int]:
    gc.collect()
    tracemalloc.start()
    try:
        result = operation()
        _, peak = tracemalloc.get_traced_memory()
        return result, peak
    finally:
        tracemalloc.stop()


memory_results = []
with TemporaryDirectory() as directory:
    data_path = Path(directory) / "measurements.txt"
    for repetitions in (1_000, 10_000):
        with data_path.open("w", encoding="utf-8") as file:
            streaming.write_sample(file, repetitions)
        eager_result, eager_peak = measure_peak(lambda: summarize_eager(data_path))
        lazy_result, lazy_peak = measure_peak(lambda: summarize_streaming(data_path))
        assert eager_result == lazy_result == (2 * repetitions, 29.5)
        memory_results.append((4 * repetitions, eager_peak, lazy_peak))

print(f"{'Records':>10} {'List peak (KiB)':>18} {'Stream peak (KiB)':>20}")
for records, eager_peak, lazy_peak in memory_results:
    print(f"{records:>10} {eager_peak / 1024:>18.1f} {lazy_peak / 1024:>20.1f}")

### Ejercicio 8 · Interpretar la medición

Compara los dos tamaños de archivo. Calcula la razón entre picos de memoria de
lista y streaming para cada tamaño. Explica qué objetos conserva cada versión.
No conviertas una razón concreta en un resultado universal ni confundas memoria
con velocidad.

In [ ]:
# Calculate the ratios from memory_results and explain the measurements.

## 10. Práctica independiente

Amplía el procesamiento con una selección por estación. Mantén el consumo dentro
del context manager y conserva el número de línea en los errores de formato.

### Ejercicio 9 · Seleccionar una estación

Escribe `select_station(measurements, station)` como generador. Componlo con la
selección por valor y el resumen. Para las líneas de `raw_lines`, estación `north`
y mínimo 20, el resultado debe ser `(1, 23.0)`. Comprueba una estación ausente:
el resultado debe ser `(0, None)`.

In [ ]:
# Define select_station and combine it with the existing processing functions.

### Ejercicio 10 · Archivo completo, vacío e inválido

Crea tres archivos temporales: uno con `raw_lines`, uno vacío y otro con una
segunda línea inválida. Procesa los dos primeros con `open_measurements` y compara
sus resúmenes. En el tercero, captura el error fuera del `with` y comprueba la línea
indicada. Como comprobación adicional del cierre, usa `open_text`, conserva la
referencia al archivo y revisa `file.closed` después del error.

In [ ]:
# Check valid, empty and invalid files without using a permanent path.

## Revisión de la práctica

- Las funciones producen resultados bajo demanda y no acumulan el archivo completo.
- La lista vacía devuelve cantidad cero y promedio `None`.
- El resultado numérico coincide con la implementación que materializa los registros.
- La lectura ocurre dentro de un `with` y el cierre se conserva ante errores.
- Las excepciones indican la línea inválida y no se silencian accidentalmente.

### Ampliación: `yield from`

`yield from` puede delegar el recorrido a otro iterable. En este ejemplo permite
concatenar dos secuencias sin construir una lista combinada.

In [ ]:
def chain_values(first: Iterable[int], second: Iterable[int]) -> Iterator[int]:
    yield from first
    yield from second


assert list(chain_values(range(3), [10, 20])) == [0, 1, 2, 10, 20]

## 11. Corrutinas, `async` y `await`

Al leer una respuesta de red, un programa puede pasar tiempo esperando. Durante una
espera compatible con `asyncio`, el **event loop** puede ejecutar otras tareas listas
para continuar. Esa coordinación es cooperativa: una operación bloqueante o un cálculo
largo dentro de una corrutina ocupa el hilo y frena a las demás.

`async def` define una función corrutina. Llamarla crea un objeto corrutina;
`await` permite ejecutarlo y obtener su resultado. `await` no crea por sí solo una
segunda tarea: una llamada puede seguir dependiendo de la anterior.

La siguiente función representa una lectura con latencia mediante `asyncio.sleep`.
La pausa simula la espera de E/S; el diccionario contiene las respuestas del ejemplo.

In [ ]:
import asyncio
from time import perf_counter

STATION_VALUES = {"north": 18.0, "south": 27.0, "west": 32.0}


async def fetch_reading(station: str, delay: float = 0.1) -> tuple[str, float]:
    print(f"Requesting {station}")
    await asyncio.sleep(delay)
    value = STATION_VALUES[station]
    print(f"Received {station}")
    return station, value


reading = await fetch_reading("north")
assert reading == ("north", 18.0)

### Notebook y script

Jupyter/IPython permite `await` directamente en una celda porque su kernel ya ejecuta
un event loop. En un archivo `.py`, define una corrutina principal y usa
`asyncio.run(main())` en el punto de entrada. No ejecutes `asyncio.run()` dentro de un
loop activo ni anides loops para adaptar este ejemplo a la notebook.

Este fragmento es la forma de entrada de un **script**; reutiliza `fetch_reading`:

```python
async def main() -> None:
    reading = await fetch_reading("north")
    print(reading)


if __name__ == "__main__":
    asyncio.run(main())
```

### Ejercicio 11 · Dos esperas

Predice el orden de los mensajes al ejecutar dos `await fetch_reading(...)`
consecutivos. ¿Se solicita la segunda estación antes de recibir la primera?
Ejecuta el código y explica qué cambiaría si solo escribieras `fetch_reading("north")`
sin `await`. No dejes corrutinas sin consumir en la notebook.

In [ ]:
# Await north and south consecutively and inspect the message order.

## 12. Ejecución secuencial y tareas concurrentes

Primero obtenemos las lecturas una por una. Después creamos una tarea por estación
con `TaskGroup`. Cada tarea puede esperar mientras otra avanza. Al salir del
`async with`, el grupo espera a que terminen sus tareas.

`task.result()` se consulta **después** de salir del grupo. La lista de tareas conserva
el orden de creación, que puede ser diferente del orden de finalización.

In [ ]:
stations = ["north", "south", "west"]


async def collect_sequential(stations: list[str]) -> list[tuple[str, float]]:
    readings = []
    for station in stations:
        readings.append(await fetch_reading(station))
    return readings


async def collect_concurrent(stations: list[str]) -> list[tuple[str, float]]:
    async with asyncio.TaskGroup() as group:
        tasks = [group.create_task(fetch_reading(station)) for station in stations]
    return [task.result() for task in tasks]


start = perf_counter()
sequential_readings = await collect_sequential(stations)
sequential_seconds = perf_counter() - start

start = perf_counter()
concurrent_readings = await collect_concurrent(stations)
concurrent_seconds = perf_counter() - start

assert sequential_readings == concurrent_readings
print(f"Sequential: {sequential_seconds:.3f} s")
print(f"Concurrent: {concurrent_seconds:.3f} s")

Con tres pausas de 0.1 s, la parte de espera secuencial suma aproximadamente 0.3 s;
concurrencia permite solaparlas. Los tiempos reales incluyen planificación y trabajo
adicional. No usaremos un umbral de velocidad como prueba de corrección.

En el event loop habitual, una tarea ejecuta código Python a la vez. Esta técnica
no reparte un cálculo intensivo entre núcleos. Tampoco vuelve asíncronas a
`time.sleep()`, `requests.get()` o `Path.read_text()` por colocarlas dentro de `async def`.

### `create_task`, `gather` y dependencias

`asyncio.create_task()` programa una corrutina y devuelve una tarea que debemos
conservar y esperar. `gather()` permite reunir resultados en orden de entrada:

```python
readings = await asyncio.gather(*(fetch_reading(name) for name in stations))
```

Con su configuración predeterminada, si una operación de `gather` falla, el primer
error se propaga y las otras operaciones no se cancelan automáticamente.
`TaskGroup` ofrece una política distinta: ante un fallo ordinario, cancela a las
hermanas, espera su limpieza y propaga los errores agrupados.

Solo hacemos concurrentes operaciones independientes. Si hay que crear un registro
antes de consultarlo, se espera la creación antes de iniciar esa consulta.

### Ejercicio 12 · Resultado y orden

Usa `TaskGroup` para solicitar `west`, `north` y `south`. Asigna pausas diferentes y
comprueba que la lista final mantiene ese orden. Luego prueba una lista vacía.
Explica por qué el orden de impresión puede ser distinto al de los resultados.

In [ ]:
# Collect readings with different delays, then check an empty input.

## 13. Cancelación y tiempos límite

Cancelar solicita que una tarea se interrumpa; no la elimina instantáneamente.
La tarea recibe `CancelledError` en una oportunidad de suspensión y puede ejecutar
su `finally`. Después de llamar a `cancel()`, esperamos la tarea para completar su
limpieza. Si se captura `CancelledError` dentro de ella, normalmente debe relanzarse.

El evento del ejemplo permite cancelar cuando sabemos que la tarea ya comenzó.

In [ ]:
async def wait_for_update(started: asyncio.Event, events: list[str]) -> None:
    try:
        events.append("started")
        started.set()
        await asyncio.Event().wait()
    finally:
        events.append("cleaned")


started = asyncio.Event()
events = []
pending = asyncio.create_task(wait_for_update(started, events))
await started.wait()
pending.cancel()
try:
    await pending
except asyncio.CancelledError:
    print("Update cancelled")
assert events == ["started", "cleaned"]

### `asyncio.timeout`

Un límite delimita cuánto estamos dispuestos a esperar. `asyncio.timeout()` usa
cancelación y transforma la suya en `TimeoutError` al salir; capturamos ese error
**fuera** del bloque. El ejemplo espera un evento que nadie activa.
El timeout tampoco puede interrumpir a tiempo una función que bloquee el event loop.

In [ ]:
timeout_events = []
try:
    async with asyncio.timeout(0.02):
        try:
            await asyncio.Event().wait()
        finally:
            timeout_events.append("cleaned")
except TimeoutError:
    print("Deadline reached")
assert timeout_events == ["cleaned"]

### Un fallo dentro del grupo

`except*` permite atender errores de un tipo dentro de un grupo de excepciones.
Aquí una tarea falla cuando su compañera ya está esperando. El grupo cancela esa
compañera y espera a que ejecute `finally` antes de propagar `ValueError`.

In [ ]:
failure_started = asyncio.Event()
failure_events = []


async def reject_reading() -> None:
    await failure_started.wait()
    raise ValueError("Invalid reading")


try:
    async with asyncio.TaskGroup() as group:
        waiting_task = group.create_task(
            wait_for_update(failure_started, failure_events)
        )
        group.create_task(reject_reading())
except* ValueError as errors:
    print(f"Failed operations: {len(errors.exceptions)}")

assert waiting_task.cancelled()
assert failure_events == ["started", "cleaned"]

### Ejercicio 13 · Limpieza ante un timeout

Aplica un timeout a `wait_for_update` y comprueba que `events` contiene `"cleaned"`.
Captura `TimeoutError` fuera del bloque y explica por qué capturarlo dentro no
representa el mismo flujo. No sustituyas el fallo por un resultado exitoso.

In [ ]:
# Use a deadline and verify cleanup after TimeoutError.

## 14. Context managers asíncronos

Un cliente de red puede necesitar esperar al abrir o cerrar sus recursos.
`async with` usa `__aenter__` y `__aexit__`, que se esperan con `await`.
`@asynccontextmanager` expresa ese ciclo de vida mediante un generador asíncrono
con un solo `yield` y limpieza en `finally`.

Construimos un pequeño cliente de lecturas en memoria. Sus pausas representan el
establecimiento de una sesión, cada solicitud y el cierre. `closed` permite observar
el ciclo de vida y rechazar usos posteriores al cierre.

In [ ]:
from collections.abc import AsyncIterator
from contextlib import asynccontextmanager, aclosing


class ReadingClient:
    def __init__(self) -> None:
        self.closed = True

    async def connect(self) -> None:
        await asyncio.sleep(0)
        self.closed = False

    async def fetch(self, station: str) -> tuple[str, float]:
        if self.closed:
            raise RuntimeError("Client is closed")
        return await fetch_reading(station)

    async def close(self) -> None:
        await asyncio.sleep(0)
        self.closed = True


@asynccontextmanager
async def open_reading_client() -> AsyncIterator[ReadingClient]:
    client = ReadingClient()
    await client.connect()
    try:
        yield client
    finally:
        await client.close()

In [ ]:
async with open_reading_client() as client:
    async with asyncio.TaskGroup() as group:
        tasks = [group.create_task(client.fetch(name)) for name in stations]
    client_readings = [task.result() for task in tasks]
    assert not client.closed

assert client.closed
assert client_readings == sequential_readings

El grupo está **dentro** del contexto del cliente: sus tareas terminan antes del
cierre del recurso que utilizan. `async with` permite esperar la entrada y salida;
no hace concurrente, por sí solo, el código de su bloque.

### Ejercicio 14 · Error y cierre

Lanza `ValueError("Invalid batch")` dentro de `open_reading_client()`. Comprueba
fuera del bloque que el cliente está cerrado y que el error se propagó.
Después intenta `await client.fetch("north")` y comprueba que falla.

In [ ]:
# Verify closure, error propagation, and rejection after closure.

## 15. Generadores asíncronos y `async for`

Una función `async def` que contiene `yield` produce un generador asíncrono.
Puede esperar antes de entregar el siguiente elemento. `async for` consume sus
valores uno por uno; **no** crea una tarea concurrente por elemento.

El siguiente generador abre el cliente al empezar a consumirse. Si interrumpimos
el recorrido, `aclosing()` espera su `aclose()` y ejecuta la salida del contexto.
Es la versión asíncrona del problema de consumo parcial que ya estudiamos.

In [ ]:
async def stream_readings(
    stations: list[str], events: list[str]
) -> AsyncIterator[tuple[str, float]]:
    try:
        async with open_reading_client() as client:
            for station in stations:
                reading = await client.fetch(station)
                events.append(f"received {station}")
                yield reading
    finally:
        events.append("stream closed")


stream_events = []
async with aclosing(stream_readings(stations, stream_events)) as readings:
    async for reading in readings:
        print(reading)
        break

assert stream_events == ["received north", "stream closed"]

El registro final está en `finally`: se ejecuta también al cerrar un generador
suspendido. En este recorrido solo se solicitó una lectura; el cierre ocurre
al salir de `aclosing`, antes de continuar con la siguiente instrucción.

### Ejercicio 15 · Consumo completo y parcial

Consume todas las estaciones con `async for` y comprueba los eventos recibidos.
Después interrumpe tras dos lecturas usando `aclosing`. Comprueba que no se
solicitó `west` y que el último evento es `"stream closed"`.

In [ ]:
# Compare complete consumption with an early stop after two readings.

## 16. Integración y práctica de concurrencia

Procesaremos una lista pequeña de estaciones de forma concurrente, usando el
cliente durante toda la vida del grupo. Un `Semaphore` limita cuántas solicitudes
pueden entrar a la vez. Es un límite de **operaciones activas**, no una cola que
limite la cantidad total de tareas creadas: para millones de entradas harían falta
trabajadores y una cola acotada, tema que queda fuera de esta práctica.

La siguiente celda guarda el módulo reutilizable `async_readings.py`. Como con
`streaming.py`, guarda en otro archivo tus cambios antes de volver a ejecutarla.

In [ ]:
%%writefile async_readings.py
"""Collect station readings with explicit asynchronous resource ownership."""

import asyncio
from collections.abc import AsyncIterator, Sequence
from contextlib import asynccontextmanager
from math import isfinite

STATION_VALUES = {"north": 18.0, "south": 27.0, "west": 32.0}


class ReadingClient:
    """Model an asynchronous client using in-memory readings and simulated latency."""

    def __init__(self, delay: float = 0.1) -> None:
        if not isfinite(delay) or delay < 0:
            raise ValueError("Delay must be finite and nonnegative")
        self.delay = delay
        self.closed = True
        self.active = 0
        self.peak_active = 0

    async def connect(self) -> None:
        await asyncio.sleep(0)
        self.closed = False

    async def fetch(self, station: str) -> tuple[str, float]:
        if self.closed:
            raise RuntimeError("Client is closed")
        self.active += 1
        self.peak_active = max(self.peak_active, self.active)
        try:
            await asyncio.sleep(self.delay)
            return station, STATION_VALUES[station]
        finally:
            self.active -= 1

    async def close(self) -> None:
        await asyncio.sleep(0)
        self.closed = True


@asynccontextmanager
async def open_reading_client(delay: float = 0.1) -> AsyncIterator[ReadingClient]:
    client = ReadingClient(delay)
    await client.connect()
    try:
        yield client
    finally:
        await client.close()


async def collect_readings(
    client: ReadingClient, stations: Sequence[str], limit: int = 2
) -> list[tuple[str, float]]:
    """Preserve input order and bound active requests for a small batch."""
    if isinstance(limit, bool) or not isinstance(limit, int) or limit < 1:
        raise ValueError("Limit must be a positive integer")
    semaphore = asyncio.Semaphore(limit)

    async def fetch_limited(station: str) -> tuple[str, float]:
        async with semaphore:
            return await client.fetch(station)

    async with asyncio.TaskGroup() as group:
        tasks = [group.create_task(fetch_limited(station)) for station in stations]
    return [task.result() for task in tasks]


async def main() -> None:
    async with open_reading_client() as client:
        readings = await collect_readings(client, ["north", "south", "west"])
        print(readings)
        print(f"Peak active requests: {client.peak_active}")
    print(f"Client closed: {client.closed}")


if __name__ == "__main__":
    asyncio.run(main())

In [ ]:
import async_readings

async_readings = importlib.reload(async_readings)

async with async_readings.open_reading_client(delay=0.02) as client:
    readings = await async_readings.collect_readings(client, stations, limit=2)
    assert 1 <= client.peak_active <= 2
    assert client.active == 0
assert readings == sequential_readings
assert client.closed
print(readings)

### Ejercicio 16 · Informe concurrente

Escribe `async def build_station_report(stations, limit=2)` usando el módulo.
Abre un cliente, obtiene las lecturas y devuelve un diccionario con `count`,
`average` y `peak_active`. Reutiliza `summarize_measurements` de `streaming`.

Comprueba:

- `north`, `south`, `west`: cantidad 3 y promedio `77 / 3`.
- Lista vacía: cantidad 0, promedio `None` y pico de actividad 0.
- `limit=1`: como máximo una solicitud activa.
- Límite 0: `ValueError`.
- Estación desconocida: error propagado y cliente cerrado tras salir del contexto.

Conserva las dependencias en secuencia: no calcules el resumen antes de reunir las
lecturas. Explica qué parte de esta función se beneficia de concurrencia.

### Extensión · E/S bloqueante existente

`asyncio.to_thread()` permite esperar una función bloqueante ejecutada en otro hilo.
La siguiente lectura usa un archivo pequeño solo para mostrar la integración.
Para un cálculo intensivo en Python, moverlo a un hilo no garantiza aceleración.
Cancelar la espera tampoco detiene por la fuerza la función que ya corre en ese hilo.

In [ ]:
with TemporaryDirectory() as directory:
    path = Path(directory) / "notes.txt"
    path.write_text("north;south;west", encoding="utf-8")
    text = await asyncio.to_thread(path.read_text, encoding="utf-8")
    assert text == "north;south;west"

### Revisión de concurrencia

- Todas las tareas creadas se esperan y sus errores se atienden o propagan.
- Los recursos permanecen abiertos mientras las tareas los utilizan.
- Las operaciones dependientes conservan su orden.
- La cancelación permite ejecutar la limpieza.
- Las mediciones de tiempo son observaciones, no pruebas de corrección.
- `async for` consume una secuencia; la concurrencia se introduce explícitamente.

## Referencias

- [Python: iterables y generadores](https://docs.python.org/3.13/howto/functional.html).
- [Python: contextlib](https://docs.python.org/3.13/library/contextlib.html).
- [Python: la sentencia with](https://docs.python.org/3.13/reference/compound_stmts.html#the-with-statement).
- [Python: tracemalloc](https://docs.python.org/3.13/library/tracemalloc.html).

- [Python: corrutinas y tareas](https://docs.python.org/3.13/library/asyncio-task.html).
- [Python: ejecución de asyncio](https://docs.python.org/3.13/library/asyncio-runner.html).
- [Python: semáforos](https://docs.python.org/3.13/library/asyncio-sync.html#asyncio.Semaphore).
- [IPython: autoawait](https://ipython.readthedocs.io/en/stable/interactive/autoawait.html).
